Primeiramente irei fazer algumas verificações nas colunas da tabela de satisfacao

In [0]:
from pyspark.sql import functions as F

df_bronze_satisfacao = spark.table("projeto.bronze.pesquisa_satisfacao") 

total_linhas = df_bronze_satisfacao.count()
total_chamados_unicos = df_bronze_satisfacao.select("id_chamado").distinct().count()
total_pesquisas_unicas = df_bronze_satisfacao.select("id_pesquisa").distinct().count()
qtd_duplicatas_chamado = total_linhas - total_chamados_unicos

qtd_nulos_nota = df_bronze_satisfacao.filter(F.col("nota_atendimento").isNull()).count()

df_bronze_satisfacao_outliers = df_bronze_satisfacao.filter(
    (F.col("nota_atendimento") < 1) | 
    (F.col("nota_atendimento") > 5)
)
qtd_outliers = df_bronze_satisfacao_outliers.count()
df_bronze_satisfacao_dados_incompletos = df_bronze_satisfacao.filter(
    F.col("id_chamado").isNull() |
    F.col("id_pesquisa").isNull() |
    F.col("nota_atendimento").isNull() |
    F.col("ingestion_timestamp").isNull()
)

print("RESUMO DE QUALIDADE DE DADOS")
print(f"Total de Linhas:{total_linhas}")
print(f"Chamados Únicos:{total_chamados_unicos}")
print(f"IDs Pesquisa Únicos:{total_pesquisas_unicas}")
print(f"Duplicatas de Chamado:{qtd_duplicatas_chamado}")
print(f"Notas Nulas:{qtd_nulos_nota}")
print(f"Notas Outliers (<1 ou >5):{qtd_outliers}")
print(f"Registros Incompleto{df_bronze_satisfacao_dados_incompletos.count()}")

if df_bronze_satisfacao_dados_incompletos.count() > 0:
    print("Visualizando amostra de dados incompletos:")
    display(df_bronze_satisfacao_dados_incompletos.limit(5))

if qtd_outliers > 0:
    print("Visualizando amostra de outliers:")
    display(df_bronze_satisfacao_outliers.limit(5))

PERCEBO QUE EXISTEM MUITOS VALORES DE NOTA_ATENDIMENTO FALTANTE (FAZ SENTIDO POIS OS CLIENTES NÃO SÃO OBRIGADOS A ATRIBUIR UMA NOTA AO FUNCIONÁRO), MAS OS CAMPOS DE ID ESTÃO SEMPRE COMO ÚNICOS POR ENQUANTO, MAS PRECISO GARANTIR NO MEU JOB QUE ESSAS LINHAS NÃO TERÃO DUPLICATAS

AGORA IREI FAZER UM CAST NOS MEUS DADOS

In [0]:
df_bronze_satisfacao = (
    df_bronze_satisfacao
    .withColumn("id_chamado", F.col("id_chamado").cast("int"))
    .withColumn("id_pesquisa", F.col("id_pesquisa").cast("int"))
    .withColumn("nota_atendimento", F.col("nota_atendimento").cast("int"))
    .withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("timestamp"))
)

In [0]:
df_bronze_satisfacao.write.mode("overwrite").saveAsTable("projeto.silver.pesquisa_satisfacao")